<a href="https://colab.research.google.com/github/mthudangn/llm-therapy-fidelity/blob/experiment/notebooks/thesis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# I. Data Exploration

In [ ]:
from google.colab import files

uploaded = files.upload()

## Import libraries and load dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
DATA_PATH = "AnnoMI-full.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

## Inspect data

In [ ]:
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())

In [ ]:
display(df.sample(min(5, len(df)), random_state=42))

In [ ]:
candidate_cols = [
    "transcript_id",
    "utterance_id",
    "interlocutor",
    "utterance_text",
    "main_therapist_behaviour",
    "client_talk_type",
    "mi_quality"
]

existing_cols = [c for c in candidate_cols if c in df.columns]

print("Existing key columns:")
for c in existing_cols:
    print("-", c)

## Value counts for important categorical columns

In [ ]:
for col in ["interlocutor", "main_therapist_behaviour", "client_talk_type", "mi_quality"]:
    if col in df.columns:
        print(f"\n===== {col} =====")
        print(df[col].value_counts(dropna=False))

## Filter therapist utterances

In [ ]:
therapist_df = df[df["interlocutor"].astype(str).str.lower() == "therapist"].copy()

print("Therapist rows shape:", therapist_df.shape)
therapist_df.head()

## Keep only rows with therapist text and therapist label

In [ ]:
therapist_df = therapist_df.dropna(subset=["utterance_text", "main_therapist_behaviour"]).copy()

print("Therapist rows with text + label:", therapist_df.shape)
print("\nTherapist behaviour labels:")
print(therapist_df["main_therapist_behaviour"].value_counts())

## Basic text cleaning

In [ ]:
therapist_df["utterance_text"] = (
    therapist_df["utterance_text"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

therapist_df = therapist_df[therapist_df["utterance_text"] != ""].copy()

print("After text cleaning:", therapist_df.shape)
therapist_df[["utterance_text", "main_therapist_behaviour"]].head(10)

## Class distribution plot

In [ ]:
label_counts = therapist_df["main_therapist_behaviour"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
label_counts.plot(kind="bar")
plt.title("Therapist Behaviour Label Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Inspect transcript-level MI quality

In [ ]:
if "mi_quality" in therapist_df.columns:
    print(therapist_df[["transcript_id", "mi_quality"]].drop_duplicates()["mi_quality"].value_counts(dropna=False))

Examples from each therapist behaviour class

In [ ]:
labels = therapist_df["main_therapist_behaviour"].dropna().unique().tolist()

for label in sorted(labels):
    print(f"\n===== Examples: {label} =====")
    sample_rows = therapist_df[therapist_df["main_therapist_behaviour"] == label][["utterance_text"]].head(5)
    for i, txt in enumerate(sample_rows["utterance_text"], 1):
        print(f"{i}. {txt}")

## Create a compact modelling dataframe

In [ ]:
model_df = therapist_df[[
    "transcript_id",
    "utterance_text",
    "main_therapist_behaviour"
]].copy()

model_df = model_df.rename(columns={
    "utterance_text": "text",
    "main_therapist_behaviour": "label"
})

print("Model dataframe shape:", model_df.shape)
display(model_df.head())

In [ ]:
print("Unique labels:", sorted(model_df["label"].unique().tolist()))
print("\nNumber of transcripts:", model_df["transcript_id"].nunique())
print("Number of utterances:", len(model_df))
print("\nNulls:")
print(model_df.isna().sum())

In [ ]:
output_path = "therapist_model_df.csv"
model_df.to_csv(output_path, index=False)

print(f"Saved cleaned file to: {output_path}")

---

# II. Data Preprocessing

Prepare a clean therapist-utterance dataset for modelling.

In [ ]:
# keep only necessary columns
prep_df = therapist_df[[
    "transcript_id",
    "utterance_text",
    "main_therapist_behaviour"
]].copy()

prep_df = prep_df.rename(columns={
    "utterance_text": "text",
    "main_therapist_behaviour": "label"
})

print(prep_df.shape)
prep_df.head()

## Remove missing and duplicate rows

In [ ]:
prep_df = prep_df.dropna(subset=["text", "label"]).copy()
prep_df["text"] = prep_df["text"].astype(str).str.strip()
prep_df = prep_df[prep_df["text"] != ""].copy()

before_dedup = len(prep_df)
prep_df = prep_df.drop_duplicates(subset=["transcript_id", "text", "label"]).copy()

print("Before dedup:", before_dedup)
print("After dedup :", len(prep_df))

## Normalize text

In [ ]:
import re

def clean_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

prep_df["text"] = prep_df["text"].apply(clean_text)

prep_df.head()

In [ ]:
# Inspect label distribution after cleaning
label_counts = prep_df["label"].value_counts()
label_counts

In [ ]:
# valid_labels = label_counts[label_counts >= 20].index
# prep_df = prep_df[prep_df["label"].isin(valid_labels)].copy()

# print(prep_df["label"].value_counts())

## Encode labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
prep_df["label_id"] = label_encoder.fit_transform(prep_df["label"])

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(label_mapping)

## Stratified train/validation/test split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    prep_df,
    test_size=0.30,
    random_state=42,
    stratify=prep_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)

In [ ]:
# Sanity check label balance across splits

print("Train label distribution:")
print(train_df["label"].value_counts(normalize=True).round(3))

print("\nValidation label distribution:")
print(val_df["label"].value_counts(normalize=True).round(3))

print("\nTest label distribution:")
print(test_df["label"].value_counts(normalize=True).round(3))

In [ ]:
X_train = train_df["text"].tolist()
y_train = train_df["label_id"].tolist()

X_val = val_df["text"].tolist()
y_val = val_df["label_id"].tolist()

X_test = test_df["text"].tolist()
y_test = test_df["label_id"].tolist()

print(len(X_train), len(X_val), len(X_test))

In [ ]:
train_df.to_csv("train_df.csv", index=False)
val_df.to_csv("val_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)

print("Saved: train_df.csv, val_df.csv, test_df.csv")

---

# III. Baseline Models

Evaluate classical machine learning models using TF-IDF features.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

## 3.1. TF-IDF vectorization

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train_vec = tfidf.fit_transform(train_df["text"])
X_test_vec = tfidf.transform(test_df["text"])

y_train = train_df["label_id"]
y_test = test_df["label_id"]

## 3.2. Logistic Regression baseline

In [ ]:
lr_model = LogisticRegression(max_iter=200)

lr_model.fit(X_train_vec, y_train)

y_pred_lr = lr_model.predict(X_test_vec)

print("Logistic Regression Accuracy:")
print(accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

## 3.3. Linear SVM baseline

In [ ]:
svm_model = LinearSVC()

svm_model.fit(X_train_vec, y_train)

y_pred_svm = svm_model.predict(X_test_vec)

print("Linear SVM Accuracy:")
print(accuracy_score(y_test, y_pred_svm))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

## 3.4. Model Comparison

In [ ]:
lr_acc = accuracy_score(y_test, y_pred_lr)
svm_acc = accuracy_score(y_test, y_pred_svm)

print("Model comparison:")
print("Logistic Regression:", lr_acc)
print("Linear SVM:", svm_acc)

---

# IV. LLM-based Fidelity Classification

This section evaluates an embedding-based approach using Sentence-BERT representations of therapist utterances. These dense semantic embeddings are used as inputs to downstream classifiers for therapist behaviour prediction.

In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

## 4.1. Load embedding model and encode text

In [ ]:
# Load embedding model
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Encode text

X_train_emb = sbert_model.encode(
    train_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

X_test_emb = sbert_model.encode(
    test_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

y_train = train_df["label_id"].values
y_test = test_df["label_id"].values

print("Train embeddings shape:", X_train_emb.shape)
print("Test embeddings shape :", X_test_emb.shape)

## 4.2. SBERT + Logistic Regression

In [ ]:
sbert_lr = LogisticRegression(max_iter=500)

sbert_lr.fit(X_train_emb, y_train)

y_pred_sbert_lr = sbert_lr.predict(X_test_emb)

print("SBERT + Logistic Regression Accuracy:")
print(accuracy_score(y_test, y_pred_sbert_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_sbert_lr, target_names=label_encoder.classes_))

## 4.3. SBERT + Linear SVM

In [ ]:
sbert_svm = LinearSVC()

sbert_svm.fit(X_train_emb, y_train)

y_pred_sbert_svm = sbert_svm.predict(X_test_emb)

print("SBERT + Linear SVM Accuracy:")
print(accuracy_score(y_test, y_pred_sbert_svm))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_sbert_svm, target_names=label_encoder.classes_))

## 4.4. Mixed model comparison

In [ ]:
results = {
    "TF-IDF + Logistic Regression": accuracy_score(y_test, y_pred_lr),
    "TF-IDF + Linear SVM": accuracy_score(y_test, y_pred_svm),
    "SBERT + Logistic Regression": accuracy_score(y_test, y_pred_sbert_lr),
    "SBERT + Linear SVM": accuracy_score(y_test, y_pred_sbert_svm),
}

print("Model Comparison")
for model_name, acc in results.items():
    print(f"{model_name}: {acc:.4f}")

In [ ]:
# Results table
results_df = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": list(results.values())
}).sort_values("Accuracy", ascending=False)

results_df

In [ ]:
# Save results
results_df.to_csv("baseline_vs_sbert_results.csv", index=False)
print("Saved: baseline_vs_sbert_results.csv")

---

# V. Evaluation and Error Analysis

This section evaluates model performance in more detail using confusion matrices and per-class metrics.
We also inspect misclassified therapist utterances to better understand model limitations.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

## 5.1. Confusion matrix for best model

In [ ]:
# Pick the best performing model from Part IV

cm = confusion_matrix(y_test, y_pred_sbert_lr)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix: SBERT + Logistic Regression")
plt.show()

## 5.2. Classification report table

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(
    y_test,
    y_pred_sbert_lr,
    target_names=label_encoder.classes_,
    output_dict=True
)
report_df = pd.DataFrame(report).transpose()
report_df

In [ ]:
# # Save evaluation results
# report_df.to_csv("classification_report_sbert_lr.csv")
# print("Saved evaluation report")

## 5.3. Inspect and analize misclassified or error examples

This part is mportant for thesis discussion.

In [ ]:
test_df = test_df.copy()
test_df["pred_label_id"] = y_pred_sbert_lr
test_df["pred_label"] = label_encoder.inverse_transform(test_df["pred_label_id"])

errors_df = test_df[test_df["label"] != test_df["pred_label"]]

print("Number of errors:", len(errors_df))
errors_df.head(10)

In [ ]:
# Show error examples
for i, row in errors_df.sample(10, random_state=42).iterrows():
    print("TEXT:", row["text"])
    print("TRUE:", row["label"])
    print("PRED:", row["pred_label"])
    print("-"*60)

In [ ]:
# Class-wise error counts
errors_df["label"].value_counts()

---

# VI. Session-level Fidelity Analysis

Transcript utterance → Label/Behaviour → Session metrics → Fidelity score

## 6.1. Aggregate predictions

In [ ]:
# combine predictions with transcript_id
results_df = pd.DataFrame({
    "transcript_id": X_test_ids,
    "text": X_test,
    "true_label": y_test,
    "pred_label": y_pred_lr  # or best model
})

results_df.head()

## 6.2. Compute behaviour distribution per session

In [ ]:
# count behaviours per transcript
session_stats = results_df.groupby(["transcript_id", "pred_label"]).size().unstack(fill_value=0)

# normalize to ratios
session_ratios = session_stats.div(session_stats.sum(axis=1), axis=0)

session_ratios.head()

## 6.3. Create fidelity metrics

In [ ]:
# avoid divide by zero
session_ratios["reflection_to_question"] = (
    session_stats.get("reflection", 0) /
    (session_stats.get("question", 1))
)

session_ratios["therapist_input_ratio"] = session_ratios.get("therapist_input", 0)

## 6.4. Fidelity scoring

In [ ]:
def compute_fidelity(row):
    score = 0

    if row["reflection_to_question"] > 1:
        score += 1

    if row["therapist_input_ratio"] < 0.5:
        score += 1

    return score

session_ratios["fidelity_score"] = session_ratios.apply(compute_fidelity, axis=1)

session_ratios.head()

In [ ]:
# Binary classification
session_ratios["high_fidelity"] = session_ratios["fidelity_score"] >= 1